# 2 — Bigram language model

**Before:** notebook **1** (tokens).

**This notebook:** count bigrams and sample — simplest next-char predictor.

**Dojo (optional):** run cells, then continue to notebook **3**.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
import torch.nn.functional as F

text = DATA.read_text(encoding="utf-8")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)


In [ ]:
# Bigram model: predict next character from previous character only.
N = vocab_size
counts = torch.zeros((N, N), dtype=torch.float32)
for t in range(len(data) - 1):
    i, j = data[t].item(), data[t + 1].item()
    counts[i, j] += 1

bigram = counts + 1  # smoothing
bigram /= bigram.sum(dim=1, keepdim=True)
print("Bigram table shape:", bigram.shape)


In [ ]:
# Sample from the bigram model
g = torch.Generator().manual_seed(0)
idx = torch.zeros(1, 1, dtype=torch.long)
for _ in range(200):
    logits = bigram[idx[:, -1]]
    idx_next = torch.multinomial(logits, num_samples=1, generator=g)
    idx = torch.cat([idx, idx_next], dim=1)

print(decode(idx.squeeze().tolist()))
